# CS-433 Project 1 — Final pipeline

This notebook reproduces our final AIcrowd submission (`submission.csv`).
Run all cells from top to bottom (about 4 minutes, mostly loading the CSV files).
The data must be in `data/dataset/` (see README). Everything is deterministic (fixed seed).


In [ ]:
import numpy as np

import preprocessing as prep
import cross_validation as cv
from helpers import load_csv_data, create_csv_submission

SEED = 1
np.random.seed(SEED)

### 1. Load data

In [ ]:
DATA_PATH = "data/dataset"

# Load dataset
x_train, x_test, y_train, train_ids, test_ids = load_csv_data(DATA_PATH)
# Load feature names (without first column 'Id')
names = prep.load_feature_names(f"{DATA_PATH}/x_train.csv")

print("train:", x_train.shape, "| test:", x_test.shape)
print(f"positive rate: {np.mean(y_train == 1):.1%}")

### 2. Preprocessing and feature engineering

All the preprocessing is in `preprocessing.py`:
1. **Cleaning**:  use of BRFSS codebook: "don't know / refused" codes (7, 9, 77, 99...) → NaN,
   "none" (88) → 0, yes/no → 1/0. Only ~65 useful variables are kept (dictionary `RULES`);
   `HAREHAB1` is removed because it leaks the label.
2. **Skip patterns**: when a question was skipped because of a previous answer, the NaN gets its logical value.
3. **Missing values and scaling**: columns with > 40 % NaN dropped, outliers clipped (1st-99th percentiles),
   median / mode imputation + "was missing" indicators, one-hot encoding, standardization.
4. **Feature engineering** (`RECOMMENDED_FEATURES`, chosen by the ablation study in `experiments_features.py`):
   one-hot of age group, BMI category and general health, age² and BMI², 4 optional survey modules,
   and all 105 pairwise products of 15 risk factors.

Important: all statistics (medians, means, ...) are computed on the **training data only**.
In the cross-validation below, the preprocessing is re-fitted on each training fold.

In [ ]:
FEATURES = prep.RECOMMENDED_FEATURES    # feature engineering options kept after the ablation
USE_STATE = prep.RECOMMENDED_USE_STATE  # _STATE one-hot: False (no gain in CV)

print(FEATURES)

### 3. Logistic regression with Newton's method

Our model is L2-regularized logistic regression (`newton.py`), trained with **Newton's method**
instead of gradient descent: same solution, but ~10 iterations instead of hundreds,
and no step size to tune. λ = 1e-4 was chosen by cross-validation
(see `results/feature_ablation.md`).

The comparison with the other models (ridge regression, logistic regression by gradient descent, SVM)
is in `experiments_baselines.py` → `results/model_comparison.md`.

In [ ]:
LAMBDA = prep.RECOMMENDED_LAMBDA  # 1e-4
K_FOLDS = 5

### 4. Cross-validation and decision threshold

Only ~9 % of the participants are positive, so with the default threshold (probability > 0.5)
the model predicts almost everyone as healthy and the F1 is poor (~0.22).
We therefore predict "sick" when $x^\top w > t$ and choose the threshold $t$ that maximizes F1.

To avoid an optimistic estimate, we use **stratified 5-fold cross-validation**:
preprocessing + model are re-fitted on each training fold, every participant gets a score from a
model that never saw them, and the threshold is chosen on these out-of-fold scores.

In [ ]:
res = cv.cv_newton(
    x_train, y_train, names,
    features=FEATURES, lambda_=LAMBDA, k=K_FOLDS, seed=SEED,
    use_state=USE_STATE, verbose=False,
)

print(f"Number of features : {res['n_features']}")
print(f"F1 per fold        : {np.round(res['f1_folds'], 4)}")
print(f"CV F1              : {res['f1_mean']:.4f} +- {res['f1_std']:.4f}")
print(f"Threshold on x.w   : {res['threshold']:.3f}  "
      f"(probability {1 / (1 + np.exp(-res['threshold'])):.1%})")

### 5. Final model and submission

The model is retrained on the **whole** training set and applied to the test set
with the threshold chosen in cross-validation.

In [ ]:
pred, w, feature_names = cv.fit_final_newton(
    x_train, y_train, x_test, names, res["threshold"],
    features=FEATURES, lambda_=LAMBDA, use_state=USE_STATE,
)

create_csv_submission(test_ids, pred, "submission.csv")
print(f"submission.csv written: {len(pred)} predictions, "
      f"{np.mean(pred == 1):.1%} predicted positive")